In [ ]:
import jax.numpy as jnp
from scipy.optimize import differential_evolution

# ==========================================
# 1. THE STEADY-STATE SOLVER : 
# ==========================================
def exact_iterative_steady_state(params, tol=1e-6, max_iter=50000, mixing=0.05, quiet=False):
    # Unpack parameters
    Ea, Eb, Ec = params['Ea'], params['Eb'], params['Ec']
    Da, Db, Dc = params['Delta_a'], params['Delta_b'], params['Delta_c']
    ka, kb, kc = params['kappa_a'], params['kappa_b'], params['kappa_c']
    Xcc, Xac, Xbc = params['Xcc'], params['Xac'], params['Xbc']

    # Initial linear guess
    na = jnp.abs(Ea / (ka/2.0 + 1j*Da))**2
    nb = jnp.abs(Eb / (kb/2.0 + 1j*Db))**2
    nc = jnp.abs(Ec / (kc/2.0 + 1j*Dc))**2

    # Self-Consistent Loop
    for i in range(max_iter):
        Da_eff = Da - Xac * nc
        Db_eff = Db - Xbc * nc
        Dc_eff = Dc - Xcc * nc - Xac * na - Xbc * nb

        na_calc = jnp.abs(Ea / (ka/2.0 + 1j*Da_eff))**2
        nb_calc = jnp.abs(Eb / (kb/2.0 + 1j*Db_eff))**2
        nc_calc = jnp.abs(Ec / (kc/2.0 + 1j*Dc_eff))**2

        error = jnp.max(jnp.array([
            jnp.abs(na_calc - na),
            jnp.abs(nb_calc - nb),
            jnp.abs(nc_calc - nc)
        ]))

        if error < tol:
            if not quiet:
                print(f"Success! Converged exactly in {i} iterations.")
            break

        na = (1.0 - mixing) * na + mixing * na_calc
        nb = (1.0 - mixing) * nb + mixing * nb_calc
        nc = (1.0 - mixing) * nc + mixing * nc_calc
    else:
        if not quiet:
            print("Warning: Did not perfectly converge.")

    # Final Complex Amplitude Calculation
    Da_eff = Da - Xac * nc
    Db_eff = Db - Xbc * nc
    Dc_eff = Dc - Xcc * nc - Xac * na - Xbc * nb

    alpha = Ea / (ka/2.0 + 1j * Da_eff)
    beta  = Eb / (kb/2.0 + 1j * Db_eff)
    gamma = Ec / (kc/2.0 + 1j * Dc_eff)

    if not quiet:
        print("=== True Self-Consistent Steady State ===")
        print(f"Photon Numbers (n): na={na:.5f}, nb={nb:.5f}, nc={nc:.5f}")

    return jnp.array([na, nb, nc]), jnp.array([alpha, beta, gamma])


# ==========================================
# 2. THE RESONANCE OPTIMIZER
# ==========================================
def find_resonant_parameters(base_params, bounds, target_transmon_level=1):
    
    def cost_function(x):
        Da_guess, Db_guess, Dc_guess, Ea_guess, Eb_guess, Ec_guess = x
        
        # Inject guesses into a temporary dictionary
        temp_params = base_params.copy()
        temp_params.update({
            'Delta_a': Da_guess, 'Delta_b': Db_guess, 'Delta_c': Dc_guess,
            'Ea': Ea_guess, 'Eb': Eb_guess, 'Ec': Ec_guess
        })
        
        # Run solver QUIETLY (no prints) with a looser tolerance for speed
        try:
            nums, _ = exact_iterative_steady_state(temp_params, tol=1e-5, max_iter=2000, mixing=0.1, quiet=True)
            na, nb, nc = nums
        except:
            return 1e6 # Huge penalty if solver fails or JAX throws NaN

        Xaa, Xbb, Xcc = temp_params['Xaa'], temp_params['Xbb'], temp_params['Xcc']
        Xab, Xac, Xbc = temp_params['Xab'], temp_params['Xac'], temp_params['Xbc']

        # Calculate exact Stark shifts (Pure JAX math)
        shift_a = -2.0*Xaa*na - Xab*nb - Xac*nc
        shift_b = -2.0*Xbb*nb - Xab*na - Xbc*nc
        shift_c = -2.0*Xcc*nc - Xac*na - Xbc*nb

        # Calculate displaced frame energies
        Ea_1 = Da_guess + shift_a
        Eb_1 = Db_guess + shift_b
        
        n = target_transmon_level
        Ec_n = n * (Dc_guess + shift_c) - (Xcc / 2.0) * n * (n - 1.0)

        # Cost: Distance between the three energy levels using JAX
        penalty = jnp.abs(Ea_1 - Eb_1) + jnp.abs(Ea_1 - Ec_n)
        
        # Cast to float so SciPy's differential_evolution can digest it safely
        return float(penalty)

    print(f"\nSearching for resonance: 1a = 1b = {target_transmon_level}c ...")
    
    # Run the global optimizer
    result = differential_evolution(cost_function, bounds, popsize=15, mutation=(0.5, 1.0), recombination=0.7)
    
    if result.success:
        print("\n=== Optimization Successful! ===")
        Da_opt, Db_opt, Dc_opt, Ea_opt, Eb_opt, Ec_opt = result.x
        
        final_params = base_params.copy()
        final_params.update({
            'Delta_a': Da_opt, 'Delta_b': Db_opt, 'Delta_c': Dc_opt,
            'Ea': Ea_opt, 'Eb': Eb_opt, 'Ec': Ec_opt
        })
        
        return final_params
    else:
        print("Optimizer failed to find a resonant solution.")
        return None

# ==========================================
# 3. SAMPLE INPUT AND EXECUTION
# ==========================================

# Your known laboratory parameters (Kerrs in MHz, Kappas guessed at 1.0)
base_params = {
    'Delta_a': -14.0,
    'Delta_b': -14.0,
    'Delta_c': 290.0,

    'Xaa': 0.0014,
    'Xbb': 0.0373,
    'Xcc': 164.0,

    'Xab': 0.0072,
    'Xac': 0.671,
    'Xbc': 3.5,

    'kappa_a': 0.0042,
    'kappa_b': 0.984,
    'kappa_c': 0.0207,

    'Ea': 5.6,
    'Eb': 3.0,
    'Ec': 215.0,
}

# Define the acceptable search ranges (Min, Max) for your equipment
search_bounds = [
    (-40.0, -10.0),    # Delta_a bounds
    (-40.0, -10.0),    # Delta_b bounds
    (-300.0, 300.0), # Delta_c bounds
    (0.0, 20.0),       # Ea drive bounds
    (0.0, 20.0),       # Eb drive bounds
    (0.0, 300.0)        # Ec drive bounds
]

# Run the optimizer targeting the 2nd transmon level (1a = 1b = 2c)
optimized_params = find_resonant_parameters(base_params, search_bounds, target_transmon_level=2)

# ==========================================
# 4. OUTPUT EXTRACTION & VERIFICATION
# ==========================================
if optimized_params is not None:
    print("\nExtracting final variables for Dynamiqs:")
    print(f"  Delta_a = {optimized_params['Delta_a']:.4f}")
    print(f"  Delta_b = {optimized_params['Delta_b']:.4f}")
    print(f"  Delta_c = {optimized_params['Delta_c']:.4f}")
    print(f"  Ea = {optimized_params['Ea']:.4f}")
    print(f"  Eb = {optimized_params['Eb']:.4f}")
    print(f"  Ec = {optimized_params['Ec']:.4f}")

    print("\nVerifying resulting energy levels by running forward solver once more...")
    
    # Run it with quiet=False so we can see the prints
    nums, amps = exact_iterative_steady_state(optimized_params, tol=1e-8, max_iter=5000, mixing=0.05, quiet=False)
    na, nb, nc = nums
    
    # Calculate energy levels manually to prove they match
    shift_a = -2.0*optimized_params['Xaa']*na - optimized_params['Xab']*nb - optimized_params['Xac']*nc
    shift_b = -2.0*optimized_params['Xbb']*nb - optimized_params['Xab']*na - optimized_params['Xbc']*nc
    shift_c = -2.0*optimized_params['Xcc']*nc - optimized_params['Xac']*na - optimized_params['Xbc']*nb
    
    Ea_1 = optimized_params['Delta_a'] + shift_a
    Eb_1 = optimized_params['Delta_b'] + shift_b
    Ec_2 = 2.0 * (optimized_params['Delta_c'] + shift_c) - optimized_params['Xcc']
    
    print("\n=== Final Resonant Energy Levels ===")
    print(f"  Energy of |1a> : {Ea_1:.4f} MHz")
    print(f"  Energy of |1b> : {Eb_1:.4f} MHz")
    print(f"  Energy of |2c> : {Ec_2:.4f} MHz")


Searching for resonance: 1a = 1b = 2c ...
